# Ensemble Methods: Bagging, Random Forests, and Boosting

*Pure Python explanation and implementation concepts — no external ML libraries.*

---

## What are Ensemble Methods?

**Ensemble methods** combine multiple "weak" models to produce a single "strong" model. The key insight is that a group of diverse, imperfect models can collectively make better predictions than any individual model alone.

### Why Ensembles Work

Consider the "Wisdom of Crowds" analogy:
- If you ask one person to estimate the number of jelly beans in a jar, they'll likely be off.
- But if you average the guesses of 100 people, the result is usually very close to the true number.

The same principle applies to machine learning:
- Individual models have different **biases** and **variances**.
- Combining them **reduces variance** (bagging) or **reduces bias** (boosting).

### The Big Three Ensemble Strategies

| Strategy | How it combines models | Primary benefit |
|---|---|---|
| **Bagging** | Train on bootstrap samples, vote/average | Reduces variance |
| **Random Forest** | Bagging + random feature subsets | Reduces variance + decorrelates trees |
| **Boosting** | Train sequentially, focus on mistakes | Reduces bias |

## Bootstrap Sampling

**Bootstrap sampling** is the foundation of bagging. It creates new datasets by **sampling with replacement** from the original dataset.

Key properties:
- Each bootstrap sample has the **same size** as the original dataset.
- On average, each sample contains about **63.2%** of the unique original instances (the rest are duplicates).
- The ~36.8% of instances *not* selected form the **out-of-bag (OOB)** set — useful for validation!

### Why 63.2%?

The probability that a specific instance is *not* picked in one draw is $(1 - 1/n)$. Over $n$ draws:

$$P(\text{not picked}) = \left(1 - \frac{1}{n}\right)^n \approx \frac{1}{e} \approx 0.368$$

So $P(\text{picked at least once}) \approx 1 - 0.368 = 0.632$.

In [ ]:
from random import seed
from random import randrange


def subsample(dataset, ratio=1.0):
    """Create a bootstrap subsample from the dataset (sampling with replacement)."""
    sample = list()
    n_sample = round(len(dataset) * ratio)
    while len(sample) < n_sample:
        index = randrange(len(dataset))
        sample.append(dataset[index])
    return sample


def mean(numbers):
    """Compute the arithmetic mean of a list of numbers."""
    return sum(numbers) / float(len(numbers))

### Demonstrating Bootstrap Convergence

As we take more bootstrap samples, the estimated mean converges to the true population mean:

In [ ]:
seed(1)
dataset = [[randrange(10)] for i in range(20)]

ratio = 0.10
for size in [1, 10, 100, 1000, 10000]:
    sample_means = list()
    for i in range(size):
        sample = subsample(dataset, ratio)
        sample_mean = mean([row[0] for row in sample])
        sample_means.append(sample_mean)
    print("Samples: [%5d], Estimated mean: [%.3f]" % (size, mean(sample_means)))

print("\nTrue dataset mean: [%.3f]" % mean([row[0] for row in dataset]))

## Bagging (Bootstrap Aggregating)

**Bagging**, introduced by Leo Breiman in 1996, is a meta-algorithm that:

1. Creates **B** bootstrap samples from the training data.
2. Trains one model (typically a decision tree) on each bootstrap sample.
3. Combines predictions:
   - **Classification**: Majority vote across all B models.
   - **Regression**: Average of all B predictions.

### Algorithm

```
Input: Training set D of size n, number of models B
For b = 1 to B:
    1. Create bootstrap sample D_b by sampling n instances with replacement from D
    2. Train model M_b on D_b
Prediction:
    Classification: return mode(M_1(x), M_2(x), ..., M_B(x))
    Regression:     return mean(M_1(x), M_2(x), ..., M_B(x))
```

### Why Bagging Reduces Variance

Each individual decision tree is a **high-variance** model (small changes in data → very different tree). By averaging B such trees trained on different bootstrap samples, the variance of the ensemble drops by roughly a factor of B — *if the trees are uncorrelated*.

This is the key limitation: bagged trees tend to be **correlated** (they all see similar data and tend to split on the same strong features). Random Forests address this.

## Random Forests

**Random Forests** (Breiman, 2001) extend bagging with one crucial addition: **random feature selection** at each split.

### How It Differs from Bagging

| Aspect | Bagging | Random Forest |
|---|---|---|
| Base learner | Decision tree (all features) | Decision tree (random subset of features) |
| Feature selection | All features at each split | Random $m$ features at each split |
| Tree correlation | High | Low |
| Typical $m$ | N/A | $\sqrt{p}$ (classification) or $p/3$ (regression) |

where $p$ is the total number of features.

### Algorithm

```
Input: Training set D, number of trees B, features per split m
For b = 1 to B:
    1. Create bootstrap sample D_b
    2. Train decision tree T_b on D_b, but at each split:
       a. Randomly select m features from the full feature set
       b. Find the best split among only those m features
       c. Split the node
Prediction:
    Classification: return mode(T_1(x), T_2(x), ..., T_B(x))
    Regression:     return mean(T_1(x), T_2(x), ..., T_B(x))
```

### Why Random Feature Selection Helps

By forcing each tree to consider only a random subset of features at each split:
- Trees become **more diverse** (decorrelated).
- The variance reduction from averaging is more effective.
- Even if one feature is very strong, not all trees will use it at the root → more robust ensemble.

### Hyperparameters

| Parameter | Description | Typical values |
|---|---|---|
| `n_trees` | Number of trees in the forest | 100–1000 |
| `max_features` | Features considered at each split | $\sqrt{p}$ for classification |
| `max_depth` | Maximum depth of each tree | None (grow fully) or tuned |
| `min_samples_split` | Minimum samples to split a node | 2–10 |
| `sample_size` | Bootstrap sample size ratio | 1.0 (full size with replacement) |

## The Voting Mechanism

### Hard Voting (Majority Vote)

Each model casts one vote for a class. The class with the most votes wins.

```python
predictions = [model.predict(x) for model in ensemble]
final = max(set(predictions), key=predictions.count)
```

### Soft Voting (Probability Averaging)

Each model outputs class probabilities. The probabilities are averaged, and the class with the highest average probability wins.

```python
avg_probs = mean([model.predict_proba(x) for model in ensemble], axis=0)
final = argmax(avg_probs)
```

Soft voting typically performs better because it accounts for the *confidence* of each model's prediction.

### Example

Suppose 5 trees predict the following for a binary classification:

| Tree | Class 0 prob | Class 1 prob | Hard vote |
|------|-------------|-------------|-----------|
| T1 | 0.9 | 0.1 | 0 |
| T2 | 0.4 | 0.6 | 1 |
| T3 | 0.3 | 0.7 | 1 |
| T4 | 0.8 | 0.2 | 0 |
| T5 | 0.45 | 0.55 | 1 |

- **Hard vote**: Class 1 wins (3 vs 2).
- **Soft vote**: avg(Class 0) = 0.57, avg(Class 1) = 0.43 → Class 0 wins!

The soft vote correctly identifies that the trees predicting Class 0 are much more confident.

In [ ]:
# Simple majority vote implementation

def majority_vote(predictions):
    """Return the class with the most votes."""
    return max(set(predictions), key=predictions.count)

# Example: 5 trees making predictions
tree_predictions = [0, 1, 1, 0, 1]
print("Individual predictions:", tree_predictions)
print("Majority vote result: ", majority_vote(tree_predictions))

## Out-of-Bag (OOB) Error Estimation

One of the most elegant features of bagging is **free validation** via the out-of-bag error.

### How It Works

1. For each bootstrap sample, ~36.8% of the original instances are **not selected** (the OOB set).
2. After training, each instance can be predicted by the trees that did *not* include it in their training set.
3. The OOB error is the prediction error computed over these OOB predictions.

### Why It Matters

- **No need for a separate validation set** — every instance eventually gets predicted by trees that haven't seen it.
- OOB error is an unbiased estimate of the test error (similar to leave-one-out cross-validation).
- Saves computation: no need for k-fold cross-validation.

### Algorithm

```
For each instance x_i in the training set:
    1. Collect predictions from all trees where x_i was OOB
    2. Aggregate (vote / average) these predictions
    3. Compare with the true label
OOB Error = fraction of incorrect OOB predictions
```

Breiman (2001) showed that the OOB error converges to the true generalization error as the number of trees increases.

## Boosting: AdaBoost Basics

While bagging trains models **in parallel** on independent bootstrap samples, **boosting** trains models **sequentially**, with each new model focusing on the mistakes of the previous ones.

### Key Idea

- Start with equal weights for all training instances.
- Train a weak learner (e.g., a decision stump — a tree with one split).
- **Increase weights** for misclassified instances.
- **Decrease weights** for correctly classified instances.
- Train the next model on the reweighted data.
- Final prediction is a **weighted vote** of all models.

### AdaBoost Algorithm

```
Input: Training set D = {(x_1, y_1), ..., (x_n, y_n)}, number of rounds T
Initialize weights: w_i = 1/n for all i

For t = 1 to T:
    1. Train weak learner h_t on D with weights w
    2. Compute weighted error:
       ε_t = Σ w_i · I(h_t(x_i) ≠ y_i) / Σ w_i
    3. Compute model weight:
       α_t = 0.5 × ln((1 - ε_t) / ε_t)
    4. Update instance weights:
       w_i = w_i × exp(-α_t · y_i · h_t(x_i))
    5. Normalize weights

Final prediction: sign(Σ α_t · h_t(x))
```

### Bagging vs. Boosting

| Aspect | Bagging | Boosting |
|---|---|---|
| Training | Parallel | Sequential |
| Sample weights | Uniform | Adaptive |
| Reduces | Variance | Bias |
| Overfitting risk | Low | Higher (can be mitigated) |
| Example | Random Forest | AdaBoost, Gradient Boosting, XGBoost |

## Putting It All Together: Random Forest in Pseudocode

Below is a concise pseudocode summarizing how a random forest works end-to-end. For full implementations, see `case08_using_bagging_to_predict_sonar_data.py` and `case09_using_random_forest_to_predict_sonar_data.py` in the Kaggle practice projects folder.

```
function random_forest_fit(train, n_trees, max_features, sample_ratio):
    forest = []
    for i in 1 to n_trees:
        sample = bootstrap_sample(train, sample_ratio)
        tree = build_tree(sample, max_features)
        forest.append(tree)
    return forest

function random_forest_predict(forest, row):
    predictions = [tree_predict(tree, row) for tree in forest]
    return majority_vote(predictions)

function build_tree(sample, max_features):
    # At each node, only consider a random subset of features
    features = random_select(all_features, max_features)
    best_feature, best_value = find_best_split(sample, features)
    # Recursively split until stopping criteria met
    ...
```

In [ ]:
# Demonstrate the subsample function used in bagging/random forests

seed(42)

dataset = [[i, i * 2, i % 2] for i in range(10)]
print("Original dataset:")
for row in dataset:
    print(" ", row)

bootstrap_sample = subsample(dataset, ratio=1.0)
print("\nBootstrap sample (same size, with replacement):")
for row in bootstrap_sample:
    print(" ", row)

# Count unique rows to verify ~63.2% coverage
unique_rows = [str(row) for row in bootstrap_sample]
print("\nUnique instances in bootstrap: %d / %d (%.1f%%)"
      % (len(set(unique_rows)), len(dataset), 100 * len(set(unique_rows)) / len(dataset)))

---

## Summary

| Method | Key Idea | Variance | Bias |
|---|---|---|---|
| **Bagging** | Bootstrap + aggregate | ↓↓ | — |
| **Random Forest** | Bagging + random features | ↓↓↓ | — |
| **Boosting** | Sequential + focus on errors | — | ↓↓ |

### Key Takeaways

1. **Ensemble methods** combine weak learners to create strong learners.
2. **Bootstrap sampling** (with replacement) is the foundation of bagging.
3. **Bagging** reduces variance by averaging predictions from models trained on different subsets.
4. **Random Forests** further reduce variance by decorrelating trees through random feature selection.
5. **OOB error** provides free validation — no need for a separate holdout set.
6. **Boosting** takes a different approach: reducing bias by focusing on hard-to-classify instances.
7. In practice, **Random Forests** and **Gradient Boosting** (XGBoost, LightGBM) are among the most effective algorithms for tabular data.

### Next Steps
- Run `case08_using_bagging_to_predict_sonar_data.py` to see bagging in action.
- Run `case09_using_random_forest_to_predict_sonar_data.py` to see random forests in action.
- Experiment with different numbers of trees and feature subset sizes.